# 03 · Calibration, decisions, explainability and fairness
Turns the ranking model into a **decision system**: calibrated probabilities of default (PD), a
profit-maximising approval threshold, per-applicant explanations (reason codes) and a fairness audit with mitigation.
This is the **only** notebook that evaluates on the holdout.

> Runtime: CPU is fine (about 15–25 min).

In [ ]:
# ---------- Setup: run this cell first ----------
import os, sys, subprocess, json, warnings
warnings.filterwarnings("ignore")
SMOKE = os.environ.get("CR_SMOKE") == "1"          # local smoke test on synthetic data (ignore on Colab)
if SMOKE:
    PROJECT_DIR = os.environ["CR_PROJECT_DIR"]
else:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/masters-projects/credit-risk"   # <- change if you put it elsewhere
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{PROJECT_DIR}/requirements-colab.txt"], check=True)
sys.path.insert(0, f"{PROJECT_DIR}/src")
ART = f"{PROJECT_DIR}/artifacts"; FIG = f"{PROJECT_DIR}/reports/figures"
os.makedirs(ART, exist_ok=True); os.makedirs(FIG, exist_ok=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
SEED = 42

def savefig(name):
    plt.tight_layout(); plt.savefig(f"{FIG}/{name}.png", dpi=160, bbox_inches="tight"); plt.show()

def save_json(obj, name):
    with open(f"{ART}/{name}.json", "w") as f: json.dump(obj, f, indent=2, default=float)
print("Project dir:", PROJECT_DIR)

In [ ]:
import lightgbm as lgb, pickle
from creditrisk.evaluate import (classification_summary, Calibrator, plot_reliability, profit_curve,
                                 best_threshold, breakeven_pd, expected_calibration_error)
from creditrisk import fairness as F, explain as E
feat = pd.read_parquet(f"{ART}/features.parquet")
FEATURES = json.load(open(f"{ART}/feature_list.json"))["features"]
oof, hop = pd.read_parquet(f"{ART}/oof_predictions.parquet"), pd.read_parquet(f"{ART}/holdout_predictions.parquet")
cvres = json.load(open(f"{ART}/cv_results.json"))
tr = feat[feat.SPLIT == "train"].set_index("SK_ID_CURR").loc[oof.SK_ID_CURR].reset_index()
ho = feat[feat.SPLIT == "holdout"].set_index("SK_ID_CURR").loc[hop.SK_ID_CURR].reset_index()
y, y_ho = oof.TARGET.values, hop.TARGET.values
MODEL = "lgbm_tuned"
boosters = [lgb.Booster(model_file=f"{ART}/lgbm_tuned_fold{i}.txt") for i in range(cvres["n_folds"])]
print(len(tr), len(ho))

## 1. Calibration
Calibrators are fit on **OOF** predictions (training split) and evaluated on the holdout.

In [ ]:
cal = {"raw": None, "platt": Calibrator("platt").fit(oof[MODEL], y), "isotonic": Calibrator("isotonic").fit(oof[MODEL], y)}
pd_ho = {k: (hop[MODEL].values if c is None else c.transform(hop[MODEL])) for k, c in cal.items()}
# Also: the re-weighted model's raw probabilities, to show why weighting hurts calibration
pd_ho["weighted_raw"] = hop["lgbm_weighted"].values
calib_table = pd.DataFrame({k: classification_summary(y_ho, v) for k, v in pd_ho.items()}).T
calib_table[["auc", "brier", "log_loss", "ece"]].round(5)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
plot_reliability(axes[0], y_ho, {k: pd_ho[k] for k in ["raw", "platt", "isotonic"]}); axes[0].set_title("Reliability (holdout)")
plot_reliability(axes[1], y_ho, {"weighted_raw": pd_ho["weighted_raw"], "isotonic": pd_ho["isotonic"]})
axes[1].set_title("Class-weighted model is badly mis-calibrated")
savefig("03_calibration")
best_cal = min(["platt", "isotonic"], key=lambda k: calib_table.loc[k, "brier"])
calibrator = cal[best_cal]; PD_OOF, PD_HO = calibrator.transform(oof[MODEL]), pd_ho[best_cal]
print("chosen calibrator:", best_cal)

## 2. Final holdout performance (first and only use of the holdout)

In [ ]:
final_rows = {"Logistic regression": hop["logreg"], "LightGBM (default)": hop["lgbm"], "XGBoost": hop["xgb"],
              "CatBoost": hop["cat"], "LightGBM (tuned)": hop["lgbm_tuned"], "Blend": hop["blend"]}
final = pd.DataFrame({k: classification_summary(y_ho, v, is_probability=False) for k, v in final_rows.items()}).T
final.loc["LightGBM (tuned) + calibration", ["brier", "ece"]] = [calib_table.loc[best_cal, "brier"], calib_table.loc[best_cal, "ece"]]
final.to_csv(f"{ART}/holdout_metrics.csv"); final.round(4)

## 3. From probabilities to decisions: a profit-maximising threshold
Simple P&L assumptions (stated, not learned): a repaid loan earns **10%** of the credit amount; a default loses
**50%** (loss given default). With calibrated PDs, theory says: approve iff PD < margin/(margin+LGD) = **0.167**.
We pick the threshold on the OOF predictions and report the holdout result.

In [ ]:
MARGIN, LGD = 0.10, 0.50
curve_tr = profit_curve(y, PD_OOF, tr.AMT_CREDIT, MARGIN, LGD)
bt = best_threshold(curve_tr); THR = bt["threshold"]
print(f"empirical best threshold (train OOF): {THR:.3f}   theoretical break-even PD: {breakeven_pd(MARGIN, LGD):.3f}")
curve_ho = profit_curve(y_ho, PD_HO, ho.AMT_CREDIT, MARGIN, LGD, thresholds=[THR, breakeven_pd(MARGIN, LGD), 1.01])
curve_ho["policy"] = ["optimised threshold", "theoretical break-even", "approve everyone"]
curve_ho

In [ ]:
fig, ax1 = plt.subplots(figsize=(7, 4))
c = profit_curve(y_ho, PD_HO, ho.AMT_CREDIT, MARGIN, LGD)
ax1.plot(c.threshold, c.profit / 1e9, label="profit (bn)"); ax1.axvline(THR, ls="--", c="k", lw=1)
ax1.set_xlabel("PD threshold (approve if PD < t)"); ax1.set_ylabel("holdout profit (bn currency units)")
ax2 = ax1.twinx(); ax2.plot(c.threshold, c.approval_rate, c="#55A868", label="approval rate"); ax2.set_ylabel("approval rate")
ax1.set_xlim(0, 0.5); ax1.set_title("Profit vs. approval threshold (holdout)"); fig.legend(loc="lower right")
savefig("03_profit_curve")

In [ ]:
# Sensitivity: how does the optimal threshold move with the business assumptions?
sens = []
for m in [0.05, 0.10, 0.15, 0.20]:
    for l in [0.3, 0.5, 0.7]:
        b = best_threshold(profit_curve(y, PD_OOF, tr.AMT_CREDIT, m, l))
        sens.append({"margin": m, "lgd": l, "best_threshold": b["threshold"], "breakeven_pd": breakeven_pd(m, l),
                     "approval_rate": b["approval_rate"]})
sens = pd.DataFrame(sens); sens.round(3)

## 4. Explainability (SHAP)

In [ ]:
import shap
sample = ho.sample(min(3000, len(ho)), random_state=SEED)
explainer, sv = E.shap_values(boosters[0], sample[FEATURES])
gimp = E.global_importance(sv, FEATURES); gimp.head(15)

In [ ]:
shap.plots.beeswarm(sv, max_display=20, show=False); plt.title("SHAP summary (holdout sample)"); savefig("03_shap_beeswarm")

In [ ]:
top3 = gimp.feature.head(3).tolist()
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, f in zip(axes, top3):
    shap.plots.scatter(sv[:, f], ax=ax, show=False)
savefig("03_shap_dependence")

In [ ]:
# Reason codes for a rejected applicant (adverse-action explanation) and an approved one
s_pd = calibrator.transform(np.mean([b.predict(sample[FEATURES]) for b in boosters], axis=0))
rej = int(np.argmax(s_pd)); acc = int(np.argmin(s_pd))
for label, i in [("REJECTED", rej), ("APPROVED", acc)]:
    print(f"=== {label} applicant {sample.SK_ID_CURR.iloc[i]}  PD={s_pd[i]:.3f} ===")
    print(E.format_reasons(E.reason_codes(sv[i], sample[FEATURES].iloc[i])), "\n")
shap.plots.waterfall(sv[rej], max_display=12, show=False); savefig("03_shap_waterfall_rejected")

## 5. Fairness audit
Groups: **gender** (not a model input) and **age band** (a model input, as in many real scorecards).
Decision: approve if PD < threshold from Section 3. Metrics are defined in `src/creditrisk/fairness.py`.

In [ ]:
approve_ho = PD_HO < THR
ho_gender = ho.CODE_GENDER.astype(str).values; ho_age = F.age_group(ho.AGE_YEARS).values
mask_g = np.isin(ho_gender, ["F", "M"])
rep_gender = F.group_report(y_ho[mask_g], PD_HO[mask_g], approve_ho[mask_g], ho_gender[mask_g], ho.AMT_CREDIT.values[mask_g])
rep_age = F.group_report(y_ho, PD_HO, approve_ho, ho_age, ho.AMT_CREDIT.values)
display(rep_gender.round(4)); display(rep_age.round(4))
disp = {"gender": F.disparity_summary(rep_gender), "age": F.disparity_summary(rep_age)}
pd.DataFrame(disp).round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, col, t in zip(axes, ["default_rate", "approval_rate", "good_reject_rate"],
                      ["Observed default rate", "Approval rate", "Creditworthy applicants rejected"]):
    rep_age[col].plot.bar(ax=ax, color="#4C72B0"); ax.set_title(t + " by age"); ax.tick_params(axis="x", rotation=0)
savefig("03_fairness_age")

### 5a. Does dropping gender make the model "blind" to it? (proxy test)
If the other features predict gender well, removing the column does not remove the information.

In [ ]:
mask_tr = tr.CODE_GENDER.astype(str).isin(["F", "M"]).values
n_proxy = min(60000, mask_tr.sum())
idx = np.random.RandomState(SEED).choice(np.where(mask_tr)[0], n_proxy, replace=False)
gender_proxy_auc = F.proxy_auc(tr[FEATURES].iloc[idx], tr.CODE_GENDER.astype(str).values[idx], seed=SEED)
print(f"Gender can be predicted from the model's features with AUC = {gender_proxy_auc:.3f}")

### 5b. Ablation: training *with* gender as a feature

In [ ]:
from creditrisk import models as M
best_params = json.load(open(f"{ART}/best_params.json"))
F_WITH = FEATURES + ["CODE_GENDER"]
r_with = M.cv_lightgbm(tr[F_WITH], y, ho[F_WITH], {**best_params, "learning_rate": 0.05}, n_folds=3, seed=SEED, name="with gender")
r_wo = M.cv_lightgbm(tr[FEATURES], y, ho[FEATURES], {**best_params, "learning_rate": 0.05}, n_folds=3, seed=SEED, name="without gender")
rows = []
for name, r in [("without gender (ours)", r_wo), ("with gender", r_with)]:
    c = Calibrator(best_cal).fit(r.oof, y); p_ho = c.transform(r.holdout)
    t = best_threshold(profit_curve(y, c.transform(r.oof), tr.AMT_CREDIT, MARGIN, LGD))["threshold"]
    rep = F.group_report(y_ho[mask_g], p_ho[mask_g], (p_ho < t)[mask_g], ho_gender[mask_g])
    rows.append({"model": name, "holdout_auc": classification_summary(y_ho, p_ho)["auc"], **F.disparity_summary(rep)})
unaware = pd.DataFrame(rows).set_index("model"); unaware.round(4)

### 5c. Mitigation: group-specific thresholds (post-processing)
* **Equal opportunity:** creditworthy applicants in every group are rejected at the same rate.
* **Demographic parity:** every group is approved at the same rate.

Thresholds are fit on the training OOF predictions and evaluated on the holdout.
*Caveat:* group-specific thresholds use the protected attribute at decision time. That is disparate treatment, which is illegal in many jurisdictions (e.g. under the US ECOA).
We report them to quantify the **cost of fairness**, not as a deployment recommendation.

In [ ]:
tr_gender = tr.CODE_GENDER.astype(str).values
m_tr = np.isin(tr_gender, ["F", "M"])
base_approve_tr = PD_OOF < THR
target_grr = (~base_approve_tr[m_tr & (y == 0)]).mean()   # overall good-reject rate
target_ar = base_approve_tr[m_tr].mean()                  # overall approval rate
policies = {
    "single threshold (baseline)": {g: THR for g in ["F", "M"]},
    "equal opportunity": F.group_thresholds(y[m_tr], PD_OOF[m_tr], tr_gender[m_tr], "good_reject_rate", target_grr),
    "demographic parity": F.group_thresholds(y[m_tr], PD_OOF[m_tr], tr_gender[m_tr], "approval_rate", target_ar),
}
rows = []
for name, th in policies.items():
    appr = F.apply_group_thresholds(PD_HO[mask_g], ho_gender[mask_g], th, THR)
    rep = F.group_report(y_ho[mask_g], PD_HO[mask_g], appr, ho_gender[mask_g], ho.AMT_CREDIT.values[mask_g], MARGIN, LGD)
    d = F.disparity_summary(rep)
    rows.append({"policy": name, "thr_F": th["F"], "thr_M": th["M"], "approval_rate": appr.mean(),
                 "profit": rep.profit.sum(), "approval_gap": d["approval_rate_gap"],
                 "good_reject_gap": d["good_reject_rate_gap"], "bad_approve_gap": d["bad_approve_rate_gap"]})
mitig = pd.DataFrame(rows).set_index("policy")
mitig["profit_change_%"] = 100 * (mitig.profit / mitig.profit.iloc[0] - 1)
mitig.to_csv(f"{ART}/fairness_mitigation.csv"); mitig.round(4)

## 6. Save final artefacts and an auto-generated results summary

In [ ]:
pickle.dump(calibrator, open(f"{ART}/calibrator.pkl", "wb"))
save_json({"threshold": THR, "margin": MARGIN, "lgd": LGD, "calibrator": best_cal, "model": MODEL}, "decision_policy")
sample[["SK_ID_CURR"] + FEATURES].head(500).to_parquet(f"{ART}/demo_applicants.parquet", index=False)
gimp.to_csv(f"{ART}/shap_global_importance.csv", index=False)
eda = json.load(open(f"{ART}/eda_summary.json")); cvsum = pd.read_csv(f"{ART}/cv_model_comparison.csv")
abl = pd.read_csv(f"{ART}/ablation_data_sources.csv")
fin = final.loc["LightGBM (tuned)"]
lines = [
 "# Results summary (auto-generated by notebook 03)", "",
 f"* Data: {eda['n_rows']:,} applicants, {eda['n_features']} features, default rate {eda['default_rate']:.2%}; "
 f"train {eda['n_train']:,} / holdout {eda['n_holdout']:,}",
 f"* **Holdout (tuned LightGBM): AUC {fin.auc:.4f}, Gini {fin.gini:.4f}, KS {fin.ks:.4f}, PR-AUC {fin.pr_auc:.4f}**",
 f"* Logistic-regression baseline holdout AUC: {final.loc['Logistic regression','auc']:.4f}; blend: {final.loc['Blend','auc']:.4f}",
 f"* Calibration ({best_cal}): Brier {calib_table.loc[best_cal,'brier']:.5f}, ECE {calib_table.loc[best_cal,'ece']:.5f} "
 f"(raw ECE {calib_table.loc['raw','ece']:.5f}; class-weighted model ECE {calib_table.loc['weighted_raw','ece']:.5f})",
 f"* Decision threshold PD < {THR:.3f} (theory: {breakeven_pd(MARGIN, LGD):.3f}); holdout profit vs approve-all: "
 f"{curve_ho.loc[0,'profit_vs_approve_all']:,.0f} at approval rate {curve_ho.loc[0,'approval_rate']:.2%} "
 f"(bad rate among approved {curve_ho.loc[0,'bad_rate_approved']:.2%} vs {y_ho.mean():.2%} overall)",
 f"* Gender proxy AUC from features: {gender_proxy_auc:.3f}",
 "", "## CV comparison", cvsum.round(4).to_markdown(index=False),
 "", "## Holdout metrics", final.round(4).to_markdown(),
 "", "## Data-source ablation", abl.round(4).to_markdown(index=False),
 "", "## Calibration", calib_table.round(5).to_markdown(),
 "", "## Threshold sensitivity", sens.round(3).to_markdown(index=False),
 "", "## Fairness: gender (single threshold)", rep_gender.round(4).to_markdown(),
 "", "## Fairness: age", rep_age.round(4).to_markdown(),
 "", "## Fairness through unawareness", unaware.round(4).to_markdown(),
 "", "## Mitigation", mitig.round(4).to_markdown(),
 "", "## Top-15 SHAP features", gimp.head(15).round(5).to_markdown(index=False),
]
open(f"{PROJECT_DIR}/reports/results_summary.md", "w").write("\n".join(lines))
print("\n".join(lines[:9]))

**Next:** download `reports/results_summary.md` and the `reports/figures/` folder and share them back to write the final report.